# Trading Strategy

Loads pre-computed predictions from the case study and simulates:
- **BUY** when model predicts UP
- **SELL** when model predicts DOWN, return ≥ +22% (profit-take), or return ≤ −8% (stop-loss)

Compares strategy P&L against a **Buy & Hold** baseline.

## Imports & Load Saved Data

In [41]:
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from pathlib import Path

from sentiment.log import setup_logging
setup_logging()

In [42]:
DATA_DIR = Path(".")  # adjust if notebooks/ is not the cwd

with open(DATA_DIR / "case_study_results.pkl", "rb") as f:
    all_results = pickle.load(f)

with open(DATA_DIR / "case_study_scalers.pkl", "rb") as f:
    all_scalers = pickle.load(f)

print(f"Loaded {len(all_results)} result entries")
print("Sample keys:", list(all_results.keys())[:3])

Loaded 180 result entries
Sample keys: [('1y', '20180101', '20181231', 'AAPL'), ('1y', '20180101', '20181231', 'MSFT'), ('1y', '20180101', '20181231', 'NVDA')]


In [43]:
INITIAL_CAPITAL = 10_000.0
PROFIT_TAKE     = 0.22     # sell if return >= +22%
STOP_LOSS       = -0.08    # sell if return <= -8%

# Folder name encodes the strategy parameters
pt_label      = f"pt{int(PROFIT_TAKE * 100)}"
sl_label      = f"sl{int(abs(STOP_LOSS) * 100)}"
TRADING_DIR   = Path(f"case_study_plots_with_news/trading_{pt_label}_{sl_label}")

PERIOD_SUBFOLDER = {"3m": "3months", "6m": "6months", "1y": "1year"}
for sub in PERIOD_SUBFOLDER.values():
    (TRADING_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"Trading plots will be saved to: {TRADING_DIR.resolve()}")

Trading plots will be saved to: C:\TFA\ProjectCode\quant-sentiment-score_mdl_260326\notebooks\case_study_plots_with_news\trading_pt22_sl8


## Configuration

In [44]:
def simulate_strategy(df, initial_capital=INITIAL_CAPITAL,
                      profit_take=PROFIT_TAKE, stop_loss=STOP_LOSS):
    """
    Simulate the trading strategy on a test-period DataFrame.
    df must have columns: date, close, pred_label
    Returns a DataFrame with columns: date, portfolio, position, entry_price, trade
    """
    capital     = initial_capital
    holding     = False
    entry_price = 0.0
    shares      = 0.0
    rows        = []

    for _, row in df.iterrows():
        price      = float(row["close"])
        pred       = int(row["pred_label"])
        trade      = "hold"
        sold_today = False

        if holding:
            ret = (price - entry_price) / entry_price
            if ret >= profit_take:
                capital    = shares * price
                holding    = False; shares = 0.0
                trade      = f"sell_profit({ret:+.1%})"
                sold_today = True
            elif ret <= stop_loss:
                capital    = shares * price
                holding    = False; shares = 0.0
                trade      = f"sell_stoploss({ret:+.1%})"
                sold_today = True
            elif pred == 0:
                capital    = shares * price
                holding    = False; shares = 0.0
                trade      = "sell_signal"
                sold_today = True

        # Only buy if not holding AND didn't just sell today
        if not holding and not sold_today and pred == 1:
            shares      = capital / price
            entry_price = price
            holding     = True; capital = 0.0
            trade       = "buy"

        portfolio = shares * price if holding else capital
        rows.append({
            "date":        row["date"],
            "portfolio":   portfolio,
            "position":    "long" if holding else "cash",
            "entry_price": entry_price if holding else None,
            "trade":       trade,
        })

    return pd.DataFrame(rows)


def buy_and_hold(df, initial_capital=INITIAL_CAPITAL):
    """Buy at first close, hold entire period."""
    first_price = float(df["close"].iloc[0])
    shares = initial_capital / first_price
    return pd.Series(
        df["close"].values * shares,
        index=pd.to_datetime(df["date"].values)
    )

## Plot Function

In [45]:
def plot_trading(symbol, df_test, start_str, end_str, save_dir):
    """Plot strategy vs buy-and-hold portfolio value and save PNG."""
    if df_test.empty:
        return

    strat_df = simulate_strategy(df_test)
    bh       = buy_and_hold(df_test)

    strat_ret = (strat_df["portfolio"].iloc[-1] / INITIAL_CAPITAL - 1) * 100
    bh_ret    = (bh.iloc[-1] / INITIAL_CAPITAL - 1) * 100

    title  = f"{symbol}_{start_str}-{end_str}"
    dates  = pd.to_datetime(strat_df["date"].values)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8),
                                    gridspec_kw={"height_ratios": [3, 1]},
                                    sharex=True)
    fig.suptitle(title, fontsize=12, fontweight="bold")

    # --- top: portfolio value ---
    ax1.plot(dates, strat_df["portfolio"].values,
             label=f"Strategy  ({strat_ret:+.1f}%)",
             color="#27ae60", linewidth=1.8)
    ax1.plot(bh.index, bh.values,
             label=f"Buy & Hold ({bh_ret:+.1f}%)",
             color="steelblue", linewidth=1.5, linestyle="--")
    ax1.axhline(INITIAL_CAPITAL, color="gray", linestyle=":",
                linewidth=0.8, alpha=0.6)
    ax1.set_ylabel("Portfolio Value ($)")
    ax1.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )
    ax1.legend(fontsize=10)
    ax1.set_title(
        f"Profit-take: +{PROFIT_TAKE:.0%}  |  Stop-loss: {STOP_LOSS:.0%}  |  "
        f"Initial: ${INITIAL_CAPITAL:,.0f}",
        fontsize=9
    )

    # Mark buy/sell points on portfolio line
    buys  = strat_df[strat_df["trade"] == "buy"]
    sells = strat_df[strat_df["trade"].str.startswith("sell", na=False)]
    ax1.scatter(pd.to_datetime(buys["date"]),  buys["portfolio"],
                marker="^", color="#27ae60", s=60, zorder=5, label="Buy")
    ax1.scatter(pd.to_datetime(sells["date"]), sells["portfolio"],
                marker="v", color="#e74c3c",  s=60, zorder=5, label="Sell")

    # --- bottom: position (long=1, cash=0) ---
    position_int = (strat_df["position"] == "long").astype(int)
    ax2.fill_between(dates, position_int,
                     step="post", color="#27ae60", alpha=0.3, label="In market")
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(["Cash", "Long"])
    ax2.set_ylabel("Position")
    ax2.set_xlabel("Date")

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    save_path = save_dir / f"{title}.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {save_path.name}")

## Run & Save All Trading Plots

In [ ]:
for (period_type, start_str, end_str, symbol), df_test in all_results.items():
    subfolder = TRADING_DIR / PERIOD_SUBFOLDER[period_type]
    plot_trading(symbol, df_test, start_str, end_str, subfolder)

print(f"\nAll trading plots saved to: {TRADING_DIR.resolve()}")

  Saved: AAPL_20180101-20181231.png
  Saved: MSFT_20180101-20181231.png
  Saved: NVDA_20180101-20181231.png
  Saved: AMZN_20180101-20181231.png
  Saved: GOOGL_20180101-20181231.png
  Saved: META_20180101-20181231.png
  Saved: TSLA_20180101-20181231.png
  Saved: AVGO_20180101-20181231.png
  Saved: PEP_20180101-20181231.png
  Saved: COST_20180101-20181231.png
  Saved: AAPL_20180701-20181231.png
  Saved: MSFT_20180701-20181231.png
  Saved: NVDA_20180701-20181231.png
  Saved: AMZN_20180701-20181231.png
  Saved: GOOGL_20180701-20181231.png
  Saved: META_20180701-20181231.png
  Saved: TSLA_20180701-20181231.png
  Saved: AVGO_20180701-20181231.png
  Saved: PEP_20180701-20181231.png
  Saved: COST_20180701-20181231.png
  Saved: AAPL_20180701-20180930.png
  Saved: MSFT_20180701-20180930.png
  Saved: NVDA_20180701-20180930.png
  Saved: AMZN_20180701-20180930.png
  Saved: GOOGL_20180701-20180930.png
  Saved: META_20180701-20180930.png
  Saved: TSLA_20180701-20180930.png
  Saved: AVGO_20180701-2018

## Summary — Strategy vs Buy & Hold

In [ ]:
rows = []
for (period_type, start_str, end_str, symbol), df_test in all_results.items():
    if df_test.empty:
        continue
    strat_df  = simulate_strategy(df_test)
    bh        = buy_and_hold(df_test)
    strat_ret = (strat_df["portfolio"].iloc[-1] / INITIAL_CAPITAL - 1) * 100
    bh_ret    = (bh.iloc[-1] / INITIAL_CAPITAL - 1) * 100
    n_trades  = (strat_df["trade"] == "buy").sum()
    rows.append({
        "period_type": period_type,
        "start":       start_str,
        "end":         end_str,
        "symbol":      symbol,
        "strategy_%":  round(strat_ret, 2),
        "bh_%":        round(bh_ret, 2),
        "outperform":  round(strat_ret - bh_ret, 2),
        "n_trades":    n_trades,
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

avg_out  = summary["outperform"].mean()
win_rate = (summary["outperform"] > 0).mean()
print(f"\nAverage outperformance vs B&H: {avg_out:+.2f}%")
print(f"Win rate (beat B&H):            {win_rate:.1%}")

# --- save summary as txt ---
summary_path = TRADING_DIR / "summary.txt"
with open(summary_path, "w") as f:
    f.write(f"Trading Strategy Summary\n")
    f.write(f"{'='*60}\n")
    f.write(f"Profit-take : +{int(PROFIT_TAKE*100)}%\n")
    f.write(f"Stop-loss   : -{int(abs(STOP_LOSS)*100)}%\n")
    f.write(f"Initial cap : ${INITIAL_CAPITAL:,.0f}\n")
    f.write(f"{'='*60}\n\n")
    f.write(summary.to_string(index=False))
    f.write(f"\n\n{'='*60}\n")
    f.write(f"Average outperformance vs B&H : {avg_out:+.2f}%\n")
    f.write(f"Win rate (beat B&H)           : {win_rate:.1%}\n")

print(f"\nSummary saved to: {summary_path}")